In [ ]:
# -*- coding: utf-8 -*-
"""[Stage 3] DPO Alignment Finetuning

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1gzIpmErjiB7Ov3XSfda75bwK86n9t-fy
"""

# -*- coding: utf-8 -*-
"""
stage_3_dpo_alignment_finetuning.py
=============================================================================
Sesuai "C. Stage 3: DPO Alignment Fine-Tuning" pada thesis Anda.

Google Colab T4 GPU — LoRA + DPO fine-tuning, sesuai Tabel I.

PENYESUAIAN DARI PAPER (konfirmasi Anda):
-----------------------------------------------------------------------------
- Base model = Qwen/Qwen2-7B-Instruct (BUKAN Meta-Llama-3.1-8B-Instruct
  seperti tertulis di paper — konsisten dgn keputusan Anda sebelumnya).
  Semua hyperparameter LoRA & DPO di Tabel I TETAP dipakai apa adanya.
- ColPali TIDAK dipakai di script ini — dataset DPO dari Stage 2 sudah
  berisi context yang di-retrieve, jadi fine-tuning di sini hanya butuh
  base LLM + dataset preference (prompt/chosen/rejected).

INPUT DATASET (CSV dari Stage 2):
-----------------------------------------------------------------------------
Stage 2 menghasilkan CSV dengan kolom-kolom berikut yang dipakai di sini:
  - context_retrieved : teks konteks hasil ColPali retrieval
  - pertanyaan        : query dari mahasiswa
  - chosen            : jawaban terbaik WITH context (grounded)
  - rejected          : jawaban WITHOUT context (halusinasi)

  Kolom lain (kode_pertanyaan, topik, score_*, sem_*, lexical_*, dll.)
  juga dibaca untuk keperluan logging & validasi, tapi tidak dipakai
  langsung untuk training DPO.

IMPLEMENTASI MENGIKUTI PERSAMAAN (6)-(7) & TABEL I:
1. DPO loss (persamaan 6): pakai TRL DPOTrainer loss_type="sigmoid" (default).
2. "p_ref diimplementasikan implisit tanpa model reference terpisah": TRL
   DPOTrainer dgn peft_config TANPA ref_model terpisah — log-prob referensi
   dihitung dgn men-disable adapter LoRA sementara (base model = reference).
3. LoRA update (persamaan 7, w = w_0 + (alpha/r)*BA): r=16, alpha=32
   (scaling factor alpha/r = 2.0).
4. Base model di-load 4-bit NF4 + double quantization (sesuai Tabel I),
   lalu prepare_model_for_kbit_training() (prasyarat teknis QLoRA).
=============================================================================
"""

# %% [Cell 0] Install dependency
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

!pip install -q --no-cache-dir \
    "transformers>=4.49.0" \
    "trl>=0.12.0" \
    "peft>=0.13.0" \
    "datasets>=2.20.0" \
    bitsandbytes \
    accelerate \
    psutil \
    pandas

In [ ]:
# %% [Cell 1] Imports & Konfigurasi
import gc
import torch
import psutil
import pandas as pd

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import DPOConfig, DPOTrainer
from google.colab import drive, userdata

# ====================== KONFIGURASI ======================
# --- Model & data ---
BASE_MODEL_ID = "Qwen/Qwen2-7B-Instruct"

# Output CSV dari Stage 2 — kolom yang dipakai: context_retrieved,
# pertanyaan, chosen, rejected
DPO_DATASET_FILE = "/content/drive/MyDrive/Thesis-Agung_Prawoto-56225110019/DPO/dpo_dataset_output.csv"

OUTPUT_DIR = "/content/drive/MyDrive/Thesis-Agung_Prawoto-56225110019/Dataset/qwen2-7b-dpo-lora-adapter"

# System prompt — HARUS SAMA dgn yang dipakai saat generate "chosen" di
# Stage 2 (Konteks-grounded), supaya model belajar preferensi yang konsisten.
SYSTEM_PROMPT_CHOSEN = "Jawab singkat & padat (min 3 kalimat) sesuai konteks yang diberikan"

# --- LoRA config (Tabel I) ---
LORA_R        = 16
LORA_ALPHA    = 32     # scaling factor alpha/r = 2.0 (persamaan 7)
LORA_DROPOUT  = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
LORA_BIAS     = "none"

# --- DPO training hyperparameter (Tabel I) ---
DPO_BETA               = 0.1
LEARNING_RATE          = 5e-5
PER_DEVICE_BATCH_SIZE  = 1
GRAD_ACCUM_STEPS       = 4     # effective batch = 1*4 = 4
NUM_EPOCHS             = 1
OPTIMIZER              = "paged_adamw_8bit"
LR_SCHEDULER           = "cosine"
WARMUP_RATIO           = 0.1
MAX_SEQ_LENGTH         = 1024   # DPOConfig.max_length (prompt + completion)
# Catatan: max_prompt_length dihapus dari DPOConfig sejak TRL >=0.13.
# Panjang prompt dikontrol via MAX_SEQ_LENGTH (total prompt + completion).

COMPUTE_DTYPE = torch.bfloat16
# ===========================================================

def clean_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def print_mem_usage(label: str = ""):
    tag = f" [{label}]" if label else ""
    ram = psutil.virtual_memory()
    print(f"  [MEM]{tag} RAM : {ram.used/1024**3:.2f} / {ram.total/1024**3:.2f} GB ({ram.percent:.0f}%)")
    if torch.cuda.is_available():
        alloc    = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved()  / 1024**3
        total    = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"  [MEM]{tag} VRAM: alloc={alloc:.2f}GB reserved={reserved:.2f}GB / total={total:.2f}GB")

try:
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

HF_TOKEN = userdata.get("HF_TOKEN")
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
print_mem_usage("awal sesi")


# %% [Cell 2] Load & validasi dataset CSV dari Stage 2
# ---------------------------------------------------------------------------
# Kolom CSV Stage 2 yang relevan untuk DPO training:
#   context_retrieved  -> konteks dokumen hasil ColPali retrieval
#   pertanyaan         -> query mahasiswa (= "user message")
#   chosen             -> jawaban grounded (WITH context, skor tertinggi)
#   rejected           -> jawaban halusinasi (WITHOUT context)
#
# Kolom tambahan yang dibaca untuk logging/validasi kualitas dataset:
#   kode_pertanyaan, topik, score_chosen, score_rejected, score_margin,
#   sem_grounding_chosen, sem_relevance_chosen, lexical_chosen, label_chosen
# ---------------------------------------------------------------------------

# Kolom wajib dari CSV Stage 2 yang harus ada
REQUIRED_COLS = {"context_retrieved", "pertanyaan", "chosen", "rejected"}

def load_dpo_csv(path: str) -> pd.DataFrame:
    """
    Baca CSV output Stage 2. Validasi kolom wajib, buang baris kosong
    pada field-field kritis, dan cetak ringkasan kualitas dataset.
    """
    df = pd.read_csv(path, dtype=str, keep_default_na=False)

    # Validasi kolom wajib
    missing = REQUIRED_COLS - set(df.columns)
    if missing:
        raise ValueError(
            f"Kolom wajib tidak ditemukan di CSV Stage 2: {missing}\n"
            f"Kolom yang tersedia: {list(df.columns)}"
        )

    n_awal = len(df)

    # Buang baris yang field kritis-nya kosong / NaN
    for col in REQUIRED_COLS:
        df[col] = df[col].str.strip()
    df = df[
        df["context_retrieved"].ne("") &
        df["pertanyaan"].ne("") &
        df["chosen"].ne("") &
        df["rejected"].ne("")
    ].reset_index(drop=True)

    n_valid = len(df)
    print(f"CSV Stage 2 dibaca: {n_awal} baris -> {n_valid} valid (dibuang {n_awal - n_valid} baris kosong)")

    # Log ringkasan skor jika kolom tersedia
    for col in ["score_chosen", "score_rejected", "score_margin"]:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce").dropna()
            if len(vals):
                print(f"  {col:20s}: mean={vals.mean():.3f}  min={vals.min():.3f}  max={vals.max():.3f}")

    if "label_chosen" in df.columns:
        print(f"  Distribusi label_chosen: {df['label_chosen'].value_counts().to_dict()}")

    return df


def to_trl_conversational(row: dict) -> dict:
    """
    Konversi satu baris CSV Stage 2 ke format conversational TRL:
      prompt   -> list pesan [system, user]
      chosen   -> list pesan [assistant (grounded)]
      rejected -> list pesan [assistant (halusinasi)]

    DPOTrainer otomatis menerapkan chat template tokenizer —
    tidak perlu manual templating.
    """
    user_msg = f"Konteks:\n{row['context_retrieved']}\n\nPertanyaan: {row['pertanyaan']}"
    return {
        "prompt": [
            {"role": "system",    "content": SYSTEM_PROMPT_CHOSEN},
            {"role": "user",      "content": user_msg},
        ],
        "chosen":   [{"role": "assistant", "content": row["chosen"]}],
        "rejected": [{"role": "assistant", "content": row["rejected"]}],
    }


raw_df      = load_dpo_csv(DPO_DATASET_FILE)
trl_data    = [to_trl_conversational(r) for r in raw_df.to_dict("records")]
train_dataset = Dataset.from_list(trl_data)

print(f"\nJumlah valid DPO pairs dari Stage 2: {len(train_dataset)}")
# >>> Angka ini mengisi "Valid DPO Pairs: TBD" di Tabel II thesis Anda.
print(train_dataset)
print("\nContoh 1 baris dataset (format TRL conversational):")
print(train_dataset[0])


# %% [Cell 3] Load base model (4-bit NF4 + double quant, sesuai Tabel I)
print(f"\nMemuat base model {BASE_MODEL_ID} (4-bit NF4, double quant)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,   # "double quantization" di Tabel I
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    low_cpu_mem_usage=True,
)
print_mem_usage("setelah load base model")

# Langkah WAJIB utk QLoRA: siapkan model 4-bit agar bisa di-training dgn
# LoRA + gradient checkpointing.
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
print_mem_usage("setelah prepare_model_for_kbit_training")


# %% [Cell 4] LoRA config (Tabel I, persamaan 7)
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias=LORA_BIAS,
    task_type="CAUSAL_LM",
)
print(f"LoRA scaling factor (alpha/r) = {LORA_ALPHA/LORA_R}")  # harus = 2.0


# %% [Cell 5] DPO training config (Tabel I, persamaan 6)
dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    beta=DPO_BETA,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    optim=OPTIMIZER,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    max_length=MAX_SEQ_LENGTH,
    bf16=(COMPUTE_DTYPE == torch.bfloat16),
    fp16=(COMPUTE_DTYPE == torch.float16),
    gradient_checkpointing=True,
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
)


# %% [Cell 6] Bangun DPOTrainer
# peft_config diberikan LANGSUNG ke DPOTrainer, TANPA ref_model terpisah ->
# implementasi "p_ref implisit": DPOTrainer otomatis (a) bungkus model dgn
# adapter LoRA, (b) hitung log-prob referensi dgn men-disable adapter sementara.
trainer_kwargs = dict(
    model=model,
    args=dpo_config,
    train_dataset=train_dataset,
    peft_config=lora_config,
)
# Nama parameter tokenizer berubah antar versi TRL -> coba keduanya.
try:
    trainer = DPOTrainer(tokenizer=tokenizer, **trainer_kwargs)
except TypeError:
    trainer = DPOTrainer(processing_class=tokenizer, **trainer_kwargs)

print_mem_usage("setelah DPOTrainer dibuat")


# %% [Cell 7] Training
print("\nMemulai DPO training...")
train_result = trainer.train()
print_mem_usage("setelah training selesai")
print(train_result)


# %% [Cell 8] Simpan adapter LoRA hasil fine-tuning
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nAdapter LoRA tersimpan di: {OUTPUT_DIR}")


# %% [Cell 9] (Opsional) Smoke-test: inferensi dengan adapter DPO
def quick_test(query: str, context: str, max_new_tokens: int = 200):
    """
    Uji cepat model setelah DPO. Gunakan query & context dari data Anda.
    Format prompt IDENTIK dgn to_trl_conversational() agar konsisten.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_CHOSEN},
        {"role": "user",   "content": f"Konteks:\n{context}\n\nPertanyaan: {query}"},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    enc = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            input_ids=enc["input_ids"],
            attention_mask=enc.get("attention_mask"),
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    return tokenizer.decode(
        out_ids[0][enc["input_ids"].shape[-1]:], skip_special_tokens=True
    ).strip()

# Contoh penggunaan — isi context & query dari data Anda:
# print(quick_test(
#     query="berapa sks konversi magang mandiri?",
#     context="<isi context_retrieved dari CSV Stage 2 di sini>"
# ))


# %% [Cell 10] (Opsional) Preview dataset Stage 2 yang dipakai training
def preview_stage2_dataset(n: int = 5):
    """Tampilkan n baris pertama CSV Stage 2 untuk verifikasi."""
    preview_cols = [
        "kode_pertanyaan", "topik", "pertanyaan",
        "score_chosen", "score_rejected", "score_margin", "label_chosen",
    ]
    available = [c for c in preview_cols if c in raw_df.columns]
    print(f"\n-- Preview {n} baris dataset Stage 2 (kolom informatif) --")
    print(raw_df[available].head(n).to_string(index=False))

preview_stage2_dataset()